In [3]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    GITHUB_USER = "SinghSahiljit44"  # Es. sahiljit
    REPO_NAME = "Weather_forecasting"
    REPO_URL = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"

    # Clona la repository se non è già stata clonata nella sessione
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

    # Cambia la directory di lavoro nella root del progetto
    os.chdir(REPO_NAME)
    
    # Installa le dipendenze necessarie nell'ambiente di Colab
    %pip install -q -r requirements.txt
    
    PROJECT_ROOT = Path.cwd()
else:
    # --- CONFIGURAZIONE LOCALE  ---
    PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f" [OK] Ambiente rilevato: {'Google Colab' if IN_COLAB else 'Locale'}")
print(f" [OK] Directory di lavoro impostata a: {Path.cwd()}")

 [OK] Ambiente rilevato: Locale
 [OK] Directory di lavoro impostata a: /home/matteo/Documents/GitHub/Weather_forecasting


In [4]:
import numpy as np
import pandas as pd
import src.utils.metrics as metrics
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from sklearn.svm import SVR, LinearSVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from src.data.preprocessing import get_split

def make_svr(kernel, params):
    if kernel == "linear":
        svr = LinearSVR(max_iter=20000, random_state=0, **params)
    else:
        svr = SVR(kernel=kernel, **params)
    return make_pipeline(StandardScaler(), svr)  # SVR needs scaled features

# SVR predicts a single value, so each lead time h gets its own fit
def svr_val_mae(kernel, params, h):
    svr = make_svr(kernel, params).fit(X_train, y_train[h])
    return metrics.get_mean_absolute_error(y_val[h], svr.predict(X_val))

split_j = get_split()
features = split_j.get_features()

# Temperature targets
split_j.define_target("t2m_mean")
(X_train, yt_train), (X_val, yt_val), (X_test, yt_test) = split_j.get(features)

# Precipitation targets
split_j.define_target("tp_mm")
(_, yp_train), (_, yp_val), (_, yp_test) = split_j.get(features)

y_train = pd.concat({"t2m_mean": yt_train, "tp_mm": yp_train}, axis=1)
y_val   = pd.concat({"t2m_mean": yt_val,   "tp_mm": yp_val},   axis=1)
y_test  = pd.concat({"t2m_mean": yt_test,  "tp_mm": yp_test},  axis=1)

# Optuna
Until now, our hyperparameters were fixed: yes, we did a small grid search, but we thought it was not enough. 

So here we are with Optuna! Here is the definition:

_Optuna is an automatic hyperparameter optimization software framework, particularly designed for machine learning. It features an imperative, define-by-run style user API. Thanks to our define-by-run API, the code written with Optuna enjoys high modularity, and the user of Optuna can dynamically construct the search spaces for the hyperparameters._

We sure love having everything ready to start doing some learning. We will skip redoing everything from scratch, let us start from the "joint" model.

Firstly, we'll need to import it (**perforza** we might add).

In [ ]:
import optuna 